In [11]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    # model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_KEY"),
    base_url=os.getenv("BASE_URL"),
)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    api_key=os.getenv("OPENAI_KEY"),
    base_url=os.getenv("BASE_URL"),
)

INPUT_FILE = "../inputs/menu_advanceAi.json"
DB_DIR = "../ChromaDB/db_menu_advance"

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:

import json

inputJson = []
with open( INPUT_FILE , "r" , encoding="utf-8") as f:
    inputJson = json.load(f)

from langchain_core.documents import Document

chunks = [
    Document(page_content=json.dumps(menu, ensure_ascii=False)) 
    for menu in inputJson
]

from libAgent.retriver import RetriverFactory

# Dense Retriever (MMR)
dense_retriever = RetriverFactory.createChromaRetriverMMR(embeddings=embeddings,dbPath=DB_DIR)

# Sparse Retriever (BM25)
bm25_retriever = RetriverFactory.createBM25RetrieverFromDocuments(chunks)


# Hybrid Retriever
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

In [7]:
res = dense_retriever.invoke("how can i change drive connection? ")

In [8]:
print( res[0].page_content)

{"id": "45", "label": "Drive_Connection", "type": "single setting where the user selects one option from a predefined list", "navigationToMenu": ["main>Full Set>Advance>Drive"], "description": "Setting the connection type of the board to the drive", "description_for_ai": "", "options": [{"value": "Parallel", "description": ""}, {"value": "Analog", "description": ""}, {"value": "MOD-HD5L", "description": ""}, {"value": "MOD-L1000A", "description": ""}]}


In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
                                PromptTemplate,
                                SystemMessagePromptTemplate,
                                HumanMessagePromptTemplate,
                                ChatPromptTemplate
                                )
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_classic import hub

prompt = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Answer in bullet points. Make sure your answer is relevant to the question and it is answered from the context only.
    Question: {question} 
    Context: {context} 
    Answer:
"""

template = ChatPromptTemplate.from_template(prompt)

def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

crg_chain = (
    RunnableParallel(
        {
            "context": hybrid_retriever | format_docs,
            "question": RunnablePassthrough()
        }
    )
    | template
    | llm
    | StrOutputParser()
)

In [14]:
res = crg_chain.invoke("how can i change drive connection?")

print( res )

- Go to menu: main > Full Set > Advance > Drive.  
- Open the setting labeled "Drive_Connection" (this sets the board-to-drive connection type).  
- Select the desired option from: Parallel, Analog, MOD-HD5L, or MOD-L1000A.
